In [32]:
import numpy as np
import os
os.makedirs("sleep_tracking_toolkit", exist_ok=True)


In [33]:
record_code = """
from .utils import quality_label, compute_sleep_score

class DailySleepRecord:
    def __init__(self, date, segments):
        self.date = date
        self.segments = segments

    def average_quality(self):
    if not self.segments:
        return 0
    q = np.array([q for _, q in self.segments], dtype=float)
    return float(np.round(q.mean(), 2))

def total_duration(self):
    if not self.segments:
        return 0
    d = np.array([d for d, _ in self.segments], dtype=float)
    return float(np.round(d.sum(), 2))


    def is_restful(self, duration_threshold=7, quality_threshold=75):
    return self.total_duration() >= duration_threshold and self.average_quality() >= quality_threshold

    def average_sleep_score(self):
        if not self.segments:
            return 0
        scores = [compute_sleep_score(d, q) for d, q in self.segments]
        return round(sum(scores) / len(scores), 2)

    def summary(self):
        avg_quality = self.average_quality()
        total_duration = self.total_duration()
        avg_sleep_score = self.average_sleep_score()
        label = quality_label(avg_quality)
        return {
            'date': self.date,
            'avg_quality': avg_quality,
            'total_duration': total_duration,
            'avg_sleep_score': avg_sleep_score,
            'quality_label': label
        }
"""
with open("sleep_tracking_toolkit/record.py", "w") as f:
    f.write(record_code)


In [34]:
utils_code = """
def quality_label(score):
    if score >= 85:
        return 'Excellent'
    elif score >= 70:
        return 'Good'
    elif score >= 50:
        return 'Fair'
    else:
        return 'Poor'

def normalize_quality(score, current_max=100):
    if current_max == 0:
        return 0
    return round(score / current_max * 100, 2)

def compute_sleep_score(duration, quality_score):
    score = min(duration / 8.0, 1.0) * 60 + quality_score * 0.4
    score = min(score, 100)
    return round(score, 2)
"""
with open("sleep_tracking_toolkit/utils.py", "w") as f:
    f.write(utils_code)


In [35]:
analytics_code = """
from .utils import compute_sleep_score

def overall_average_duration(records):
    if not records:
        return 0
    total = sum(r.total_duration() for r in records)
    count = len(records)
    return round(total / count, 2) if count else 0

def best_sleep_day(records):
    if not records:
        return None
    best = None
    best_score = float('-inf')
    for r in records:
        score = r.average_sleep_score()
        if score > best_score:
            best_score = score
            best = r.date
    return best

def detect_under_sleep_days(records, threshold):
    output = []
    for r in records:
        for d, _ in r.segments:
            if d < threshold:
                output.append(r.date)
                break
    return output

def detect_spike(durations, *, threshold=2):
    if not durations or len(durations) < 2:
        return False
    for a, b in zip(durations[:-1], durations[1:]):
        if abs(b - a) >= threshold:
            return True
    return False

def duration_trend(durations):
    if not durations or len(durations) < 2:
        return []
    trend = []
    for prev, curr in zip(durations[:-1], durations[1:]):
        if curr > prev:
            trend.append('up')
        elif curr < prev:
            trend.append('down')
        else:
            trend.append('same')
    return trend

def average_sleep_score_across_days(records):
    scores = []
    for r in records:
        for d, q in r.segments:
            scores.append(compute_sleep_score(d, q))
    if not scores:
        return 0
    return round(sum(scores) / len(scores), 2)
"""
with open("sleep_tracking_toolkit/analytics.py", "w") as f:
    f.write(analytics_code)


In [36]:
with open("sleep_tracking_toolkit/__init__.py", "w") as f:
    f.write("")


In [37]:
import sys
sys.path.append('/content')  # Colab working directory

from sleep_tracking_toolkit.record import DailySleepRecord
from sleep_tracking_toolkit.analytics import *


In [38]:
import shutil
shutil.make_archive('sleep_tracking_toolkit', 'zip', 'sleep_tracking_toolkit')

'/content/sleep_tracking_toolkit.zip'

In [39]:
from google.colab import files
files.download('sleep_tracking_toolkit.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Try to improve quality of coding (Optional for checking)**


In [40]:
if __name__ == "__main__":
    from sleep_tracking_toolkit.record import DailySleepRecord
    from sleep_tracking_toolkit.analytics import (
        overall_average_duration, best_sleep_day, detect_under_sleep_days,
        detect_spike, duration_trend, average_sleep_score_across_days
    )

    # Example dataset
    records = [
        DailySleepRecord("2025-09-01", [(3.5, 70), (4.0, 80)]),
        DailySleepRecord("2025-09-02", [(7.5, 88)]),
        DailySleepRecord("2025-09-03", [(6.0, 60), (1.0, 55)]),
    ]

    print("Summaries:")
    for r in records:
        print(r.summary())

    print("\nOverall avg duration:", overall_average_duration(records))
    print("Best sleep day:", best_sleep_day(records))
    print("Under-sleep days (<6h):", detect_under_sleep_days(records, threshold=6))
    durations = [r.total_duration() for r in records]
    print("Spike detected (>=2h):", detect_spike(durations, threshold=2))
    print("Duration trend:", duration_trend(durations))
    print("Avg sleep score across days:", average_sleep_score_across_days(records))


Summaries:
{'date': '2025-09-01', 'avg_quality': 75.0, 'total_duration': 7.5, 'avg_sleep_score': 58.12, 'quality_label': 'Good'}
{'date': '2025-09-02', 'avg_quality': 88.0, 'total_duration': 7.5, 'avg_sleep_score': 91.45, 'quality_label': 'Excellent'}
{'date': '2025-09-03', 'avg_quality': 57.5, 'total_duration': 7.0, 'avg_sleep_score': 49.25, 'quality_label': 'Fair'}

Overall avg duration: 7.33
Best sleep day: 2025-09-02
Under-sleep days (<6h): ['2025-09-01', '2025-09-03']
Spike detected (>=2h): False
Duration trend: ['same', 'down']
Avg sleep score across days: 61.24
